In [41]:
import warnings
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import RobustScaler
from pybaseball import statcast, batting_stats, playerid_reverse_lookup


import sys
sys.path.insert(0, "../../../")       
                                        
from training.xrv.fetch_statcast import fetch_statcast

# Download Data

In [24]:
statcast_df = fetch_statcast("2023-04-01", "2025-09-30")

This is a large query, it may take a moment to complete


100%|██████████| 30/30 [00:09<00:00,  3.06it/s]


This is a large query, it may take a moment to complete


100%|██████████| 31/31 [00:10<00:00,  3.09it/s]


This is a large query, it may take a moment to complete


...

100%|██████████| 31/31 [00:09<00:00,  3.37it/s]


This is a large query, it may take a moment to complete


100%|██████████| 31/31 [00:11<00:00,  2.70it/s]


This is a large query, it may take a moment to complete


100%|██████████| 30/30 [00:09<00:00,  3.23it/s]
/Users/mingsenwang/Library/Mobile Documents/com~apple~CloudDocs/data-science/fantasy/dugout-prophet/notebooks/training/xrv/../../../training/xrv/fetch_statcast.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(chunks, ignore_index=True)


In [25]:
fg_batters = batting_stats(2023, 2025, qual=50)

# Preprocess Data

In [26]:
IDmlbam = statcast_df.batter.unique()
lookup_df = playerid_reverse_lookup(IDmlbam)[['key_mlbam', 'key_fangraphs']]
# rename for clarity
lookup_df = lookup_df.rename(columns={
    'key_mlbam': 'batter',
    'key_fangraphs': 'IDfg',
    })

statcast_df = statcast_df.rename(columns = {'game_year': 'Season'})

trn_df = pd.merge(
    pd.merge(statcast_df, lookup_df, on='batter', how='left'),
    fg_batters[['IDfg', 'Season', 'xwOBA']], on=['IDfg', 'Season' ], how='left'
)

league_avg_xwoba = fg_batters['xwOBA'].groupby(fg_batters['Season']).mean()
trn_df['xwOBA'] = trn_df.apply(
    lambda row: league_avg_xwoba[row['Season']] if pd.isna(row['xwOBA']) else row['xwOBA'], axis=1
)

STUFF_COLS    = ['release_speed', 'pfx_x', 'pfx_z', 'release_spin_rate', 'release_extension']
COMMAND_COLS  = ['plate_x', 'plate_z', 'zone']
PITCH_TYPE_COL = 'pitch_type'
SEQUENCING_COLS = ['prev_pitch_type', 'pitch_number']  # pitch_number is pitch count within PA
BATTER_COL    = 'xwOBA'
TARGET        = 'delta_run_exp'

FEATURES  = STUFF_COLS + COMMAND_COLS + ['pitch_type_enc', 'prev_pitch_type_enc', 'pitch_number', 'xwOBA']
CONT_COLS = [c for c in FEATURES if c not in ['pitch_type_enc', 'prev_pitch_type_enc']]

trn_df = trn_df.sort_values(['game_pk', 'at_bat_number', 'pitch_number'])
trn_df['prev_pitch_type'] = trn_df.groupby(['game_pk', 'at_bat_number'])['pitch_type'].shift(1)
trn_df['prev_pitch_type'] = trn_df['prev_pitch_type'].fillna('START')

In [27]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# fit on combined to ensure consistent encoding
all_pitch_types = pd.concat([trn_df['pitch_type'], trn_df['prev_pitch_type']]).unique()
le.fit(all_pitch_types)

trn_df['pitch_type_enc'] = le.transform(trn_df['pitch_type'])
trn_df['prev_pitch_type_enc'] = le.transform(trn_df['prev_pitch_type'])

In [28]:
print(f"Rows starting with: {len(trn_df):,}")
trn_df = trn_df.dropna(subset=FEATURES + [TARGET])
print(f"Rows remaining: {len(trn_df):,}")

Rows starting with: 2,244,518
Rows remaining: 2,196,181


In [29]:
trn_df['game_date'] = pd.to_datetime(trn_df['game_date'])

train = trn_df[trn_df['game_date'] < '2025-04-01']
test  = trn_df[trn_df['game_date'] >= '2025-04-01']

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

Train: 1,506,780 | Test: 689,401


In [30]:
scaler = RobustScaler()
X_cont_train_scaled = scaler.fit_transform(X_train[CONT_COLS])
X_cont_test_scaled  = scaler.transform(X_test[CONT_COLS])

X_cont_train   = torch.tensor(X_cont_train_scaled, dtype=torch.float32)
X_cont_test    = torch.tensor(X_cont_test_scaled,  dtype=torch.float32)
X_pitch_train  = torch.tensor(X_train['pitch_type_enc'].values, dtype=torch.long)
X_pitch_test   = torch.tensor(X_test['pitch_type_enc'].values,  dtype=torch.long)
X_prev_train   = torch.tensor(X_train['prev_pitch_type_enc'].values, dtype=torch.long)
X_prev_test    = torch.tensor(X_test['prev_pitch_type_enc'].values,  dtype=torch.long)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test.values,  dtype=torch.float32)

dataloader = DataLoader(
    TensorDataset(X_cont_train, X_pitch_train, X_prev_train, y_train_tensor),
    batch_size=512, shuffle=True,
)
test_dataloader = DataLoader(
    TensorDataset(X_cont_test, X_pitch_test, X_prev_test, y_test_tensor),
    batch_size=512, shuffle=False,
)

n_pitch_types = len(le.classes_)

In [31]:
import copy
import itertools
import torch
import torch.nn as nn

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')


class PitchValueNet(nn.Module):
    def __init__(self, n_pitch_types, emb_dim, n_continuous, hidden_size, dropout):
        super().__init__()
        self.emb_current  = nn.Embedding(n_pitch_types, emb_dim)
        self.emb_previous = nn.Embedding(n_pitch_types, emb_dim)
        mlp_input_dim = emb_dim * 2 + n_continuous
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, x_cont, x_pitch_type, x_prev_pitch_type):
        x = torch.cat([self.emb_current(x_pitch_type),
                       self.emb_previous(x_prev_pitch_type),
                       x_cont], dim=1)
        return self.mlp(x)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_cont_b, X_pitch_b, X_prev_b, y_b in loader:
        X_cont_b, X_pitch_b, X_prev_b, y_b = (
            X_cont_b.to(device), X_pitch_b.to(device), X_prev_b.to(device), y_b.to(device)
        )
        optimizer.zero_grad()
        loss = criterion(model(X_cont_b, X_pitch_b, X_prev_b).squeeze(), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
    return total_loss / len(loader.dataset)


def eval_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X_cont_b, X_pitch_b, X_prev_b, y_b in loader:
            X_cont_b, X_pitch_b, X_prev_b, y_b = (
                X_cont_b.to(device), X_pitch_b.to(device), X_prev_b.to(device), y_b.to(device)
            )
            total_loss += criterion(model(X_cont_b, X_pitch_b, X_prev_b).squeeze(), y_b).item() * len(y_b)
    return total_loss / len(loader.dataset)


def run_config(emb_dim, hidden_size, lr, dropout, batch_size, n_epochs=50, patience=5):
    train_loader = DataLoader(
        TensorDataset(X_cont_train, X_pitch_train, X_prev_train, y_train_tensor),
        batch_size=batch_size, shuffle=True,
    )
    model = PitchValueNet(
        n_pitch_types=n_pitch_types,
        emb_dim=emb_dim,
        n_continuous=len(CONT_COLS),
        hidden_size=hidden_size,
        dropout=dropout,
    ).to(device)
    optimizer = Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_weights  = None
    epochs_no_improve = 0

    for epoch in range(n_epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss   = eval_one_epoch(model, test_dataloader, criterion, device)
        print(f"  epoch {epoch+1:2d}: train={train_loss:.4f}  val={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights  = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  early stop at epoch {epoch+1} (best val={best_val_loss:.4f})")
                break

    model.load_state_dict(best_weights)
    return best_val_loss, model

Using device: mps


In [32]:
# import random
# GRID = {
#     'emb_dim':     [4, 8, 16],
#     'hidden_size': [32, 64, 128],
#     'lr':          [1e-3, 3e-4, 1e-4],
#     'dropout':     [0.0, 0.1, 0.3],
#     'batch_size':  [256, 512, 1024],
# }

# results = []
# # itertools.product(
# #     GRID['emb_dim'], GRID['hidden_size'], GRID['lr'], GRID['dropout'], GRID['batch_size']
# # )
# for emb_dim, hidden_size, lr, dropout, batch_size in random.sample(list(itertools.product(*GRID.values())), k=30):
#     print(f"\nemb_dim={emb_dim}  hidden_size={hidden_size}  lr={lr}  dropout={dropout}  batch_size={batch_size}")
#     val_loss, model = run_config(emb_dim, hidden_size, lr, dropout, batch_size, patience=3)
#     results.append({
#         'emb_dim': emb_dim, 'hidden_size': hidden_size, 'lr': lr,
#         'dropout': dropout, 'batch_size': batch_size, 'val_loss': val_loss,
#     })

# results_df = pd.DataFrame(results).sort_values('val_loss')
# print("\n--- Grid search results ---")
# print(results_df.to_string(index=False))

# results differ little with best around 0.075, so we'll pick a simple config for final training
final_config = {
    'emb_dim': 4,
    'hidden_size': 32,
    'lr': 1e-3,
    'dropout': 0.0,
    'batch_size': 512,
}

In [33]:
val_loss, final_model = run_config(**final_config, n_epochs=100, patience=10)
print(f"\nFinal config: {final_config}  val_loss={val_loss:.4}")

  epoch  1: train=0.0508  val=0.0499
  epoch  2: train=0.0502  val=0.0498
  epoch  3: train=0.0501  val=0.0497
  epoch  4: train=0.0500  val=0.0496
  epoch  5: train=0.0500  val=0.0496
  epoch  6: train=0.0499  val=0.0496
  epoch  7: train=0.0499  val=0.0495
  epoch  8: train=0.0499  val=0.0496
  epoch  9: train=0.0499  val=0.0495
  epoch 10: train=0.0499  val=0.0495
  epoch 11: train=0.0499  val=0.0495
  epoch 12: train=0.0499  val=0.0495
  epoch 13: train=0.0499  val=0.0495
  epoch 14: train=0.0499  val=0.0495
  epoch 15: train=0.0499  val=0.0495
  epoch 16: train=0.0498  val=0.0495
  epoch 17: train=0.0498  val=0.0495
  epoch 18: train=0.0498  val=0.0495
  epoch 19: train=0.0498  val=0.0495
  epoch 20: train=0.0498  val=0.0495
  epoch 21: train=0.0498  val=0.0495
  epoch 22: train=0.0498  val=0.0495
  epoch 23: train=0.0498  val=0.0495
  epoch 24: train=0.0498  val=0.0495
  epoch 25: train=0.0498  val=0.0495
  epoch 26: train=0.0498  val=0.0495
  epoch 27: train=0.0498  val=0.0495
 

# Save Model

In [36]:
import torch, pickle, os

save_dir = "../../../pitcher_dashboard/"

os.makedirs(save_dir, exist_ok=True)
os.makedirs('artifacts', exist_ok=True)
os.makedirs('data', exist_ok=True)

torch.save(final_model.state_dict(), os.path.join(save_dir, 'artifacts/model.pt'))
pickle.dump(scaler, open(os.path.join(save_dir, 'artifacts/scaler.pkl'), 'wb'))
pickle.dump(le, open(os.path.join(save_dir, 'artifacts/le.pkl'), 'wb'))



In [ ]:
# save cached predictions
# classify SP vs RP based on median pitches per appearance
pitcher_appearance = (
    trn_df.groupby(['pitcher', 'game_pk'])['pitch_number']
    .count()
    .reset_index()
    .rename(columns={'pitch_number': 'pitches_in_game'})
)

pitcher_type = (
    pitcher_appearance.groupby('pitcher')['pitches_in_game']
    .median()
    .reset_index()
    .rename(columns={'pitches_in_game': 'median_pitches'})
)

pitcher_type['role'] = pitcher_type['median_pitches'].apply(
    lambda x: 'SP' if x >= 50 else 'RP'
)

# merge back to trn_df
import numpy as np

# first get full season predictions
# YOUR TASK: generate pred_xrv for the full dataset (not just test)
# hint: you need to build tensors from trn_df the same way as before

trn_df['game_date'] = pd.to_datetime(trn_df['game_date'])

X_full = trn_df[FEATURES]
y_full = trn_df[TARGET]

X_cont_full_scaled  = scaler.transform(X_full[CONT_COLS])

X_cont_full    = torch.tensor(X_cont_full_scaled,  dtype=torch.float32, device=device)
X_pitch_full  = torch.tensor(X_full['pitch_type_enc'].values, dtype=torch.long, device=device)
X_prev_full   = torch.tensor(X_full['prev_pitch_type_enc'].values, dtype=torch.long, device=device)


final_model.eval()
with torch.no_grad():
    pred_xrv_full = final_model(X_cont_full, X_pitch_full, X_prev_full).squeeze().cpu().numpy()

trn_df['pred_xrv'] = pred_xrv_full
trn_df = trn_df.merge(pitcher_type[['pitcher', 'role']], on='pitcher', how='left')


In [40]:
trn_df[['pitcher', 'game_pk', 'game_date', 'role', 'pred_xrv']].to_parquet(os.path.join(save_dir, 'data/cache.parquet'), index=False)
